# Step 2 — Create Genie spaces with information (Notebook 02)

Nine-step workshop: **1** data → **2** this notebook → **3** evals/benchmarks → **4** talk → **5** skills / no-eval space → **6** compare → **7** security → **8** deployment → **9** monitoring.

Creates **three** Genie spaces on the same manufacturing tables:

1. **Blank context** — tables only, minimal instructions.
2. **Configured (with examples)** — full business instructions + **sample questions** and **curated Q→SQL** from `templates/manufacturing_genie_configured.json` (primary: `genie_space_id`).
3. **Configured (no examples)** — **same** business instructions as (2) but **no** sample/curated pairs (`genie_space_id_configured_no_evals`) for A/B in notebook **06**.

Saves IDs to `workshop_config`. Next: **03_genie_evals_benchmarks**.

**Prerequisite:** Run `01_setup_data` first.

## Serverless compute

Attach **Serverless** notebook compute. The notebook uses the Databricks **SDK** and `spark` only for the final `workshop_config` table; config stays in **Python variables** (no `spark.conf.set` for custom keys).

## Configuration

Set **Catalog** and **Schema** to match **notebook 01** (your Unity Catalog location for this workshop). The widget defaults are **examples only**—replace them with your catalog and schema before running in a customer workspace.

All later notebooks use the **same** widget names so the path stays consistent.

In [ ]:
dbutils.widgets.text("catalog", "workshop_demo", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
fqn = f"{CATALOG}.{SCHEMA}"

SPACE_TITLE_BLANK = "Manufacturing workshop — blank context"
SPACE_TITLE_CONFIGURED = "Manufacturing workshop — configured"
SPACE_TITLE_CONFIGURED_NO_EX = "Manufacturing workshop — configured, no example SQL"
SPACE_DESC_BLANK = "Manufacturing tables only. Minimal instructions for A/B comparison."
SPACE_DESC_CONFIGURED = "Manufacturing quality analytics: OEE, FPY, defects, downtime, safety, equipment feedback."
SPACE_DESC_CONFIGURED_NO_EX = (
    "Same instructions as configured space but no sample questions or curated Q→SQL pairs (benchmark against full context in 03)."
)

print(f"Target data: {fqn}")


## SQL warehouse

Genie needs a **SQL warehouse**. This cell picks a running, starting, or serverless warehouse.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
warehouses = list(w.warehouses.list())
warehouse_id = None

for wh in warehouses:
    state = str(wh.state).upper() if wh.state else ""
    if state in ("RUNNING", "STARTING"):
        warehouse_id = wh.id
        print(f"Using warehouse: {wh.name} ({wh.id}) state={state}")
        break

if not warehouse_id:
    for wh in warehouses:
        if getattr(wh, "enable_serverless_compute", False) or "serverless" in (wh.name or "").lower():
            warehouse_id = wh.id
            print(f"Using serverless warehouse: {wh.name} ({wh.id})")
            break

if not warehouse_id and warehouses:
    wh = warehouses[0]
    warehouse_id = wh.id
    print(f"Using first warehouse: {wh.name} ({wh.id}) state={wh.state}")

if not warehouse_id:
    raise RuntimeError("No SQL warehouse found. Create or start one, then re-run.")


## Load configured Genie JSON

Load order: **local / Repo paths** → optional **workspace export** (widget path and `/Users/...` alias) → **embedded** copy of `templates/manufacturing_genie_configured.json` (base64 in this notebook so **Jobs always work**). Re-run `scripts/build_02_notebook.py` after editing the template to refresh the embedded copy.

In [ ]:
import base64
import json
import os

import requests
from databricks.sdk import WorkspaceClient

dbutils.widgets.text(
    "genie_json_path",
    "manufacturing_genie_configured.json",
    "Local/Repo relative path to manufacturing_genie_configured.json",
)
dbutils.widgets.text(
    "genie_json_workspace_path",
    "",
    "Workspace path for export fallback",
)

_EMBEDDED_B64 = "ewogICJ0aXRsZSI6ICJNYW51ZmFjdHVyaW5nIFF1YWxpdHkgQW5hbHl0aWNzIiwKICAiZGVzY3JpcHRpb24iOiAiRXhwbG9yZSBhdXRvbW90aXZlIG1hbnVmYWN0dXJpbmcgZGF0YSBpbmNsdWRpbmcgcHJvZHVjdGlvbiBldmVudHMsIHF1YWxpdHkgbWV0cmljcywgZGVmZWN0IHRyYWNraW5nLCBhbmQgcGxhbnQgcGVyZm9ybWFuY2UgdXNpbmcgbmF0dXJhbCBsYW5ndWFnZS4iLAogICJ0YWJsZV9pZGVudGlmaWVycyI6IFsKICAgICJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyIsCiAgICAibWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLm9wZXJhdG9ycyIsCiAgICAibWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIiwKICAgICJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucGxhbnRzIiwKICAgICJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucXVhbGl0eV9tZXRyaWNzX2RhaWx5IiwKICAgICJtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3Muc2FmZXR5X2luY2lkZW50cyIsCiAgICAibWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLmVxdWlwbWVudF9mZWVkYmFjayIKICBdLAogICJzcWxfaW5zdHJ1Y3Rpb25zIjogIkJ1c2luZXNzIENvbnRleHQ6XG5UaGlzIGRhdGFzZXQgcmVwcmVzZW50cyBwcm9kdWN0aW9uIGFuZCBxdWFsaXR5IGRhdGEgZnJvbSBhbiBhdXRvbW90aXZlIG1hbnVmYWN0dXJpbmcgb3BlcmF0aW9uIHdpdGggbXVsdGlwbGUgcGxhbnRzLCBwcm9kdWN0aW9uIGxpbmVzLCBhbmQgc2hpZnRzLiBVc2UgdGhlIHRhYmxlcyBpbiB0aGUgbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzIHNjaGVtYSB0byBhbnN3ZXIgcXVlc3Rpb25zIGFib3V0IHByb2R1Y3Rpb24gdm9sdW1lLCBxdWFsaXR5IHBlcmZvcm1hbmNlLCBkZWZlY3QgdHJhY2tpbmcsIGVxdWlwbWVudCBlZmZlY3RpdmVuZXNzLCBhbmQgc2FmZXR5LlxuXG5LZXkgTWV0cmljcyAmIFRocmVzaG9sZHM6XG4tIE9FRSAoT3ZlcmFsbCBFcXVpcG1lbnQgRWZmZWN0aXZlbmVzcyk6IFRhcmdldCA+PSA4NSUsIEdvb2QgPj0gNzUlLCBQb29yIDwgNjUlLiBTdG9yZWQgaW4gcXVhbGl0eV9tZXRyaWNzX2RhaWx5Lm9lZV9zY29yZSAoMCB0byAxIHNjYWxlLCBtdWx0aXBseSBieSAxMDAgZm9yIHBlcmNlbnRhZ2UpLlxuLSBGaXJzdCBQYXNzIFlpZWxkOiBUYXJnZXQgPj0gOTUlLCBBY2NlcHRhYmxlID49IDkwJSwgUG9vciA8IDg1JS4gU3RvcmVkIGluIHF1YWxpdHlfbWV0cmljc19kYWlseS5maXJzdF9wYXNzX3lpZWxkICgwIHRvIDEgc2NhbGUsIG11bHRpcGx5IGJ5IDEwMCBmb3IgcGVyY2VudGFnZSkuXG4tIERlZmVjdCBSYXRlOiBMb3cgPCAyJSwgTWVkaXVtIDItNSUsIEhpZ2ggPiA1JS4gQ2FsY3VsYXRlZCBhcyAobnVtYmVyIG9mIGRlZmVjdF9kZXRlY3RlZCBldmVudHMgLyBudW1iZXIgb2YgdW5pdF9wcm9kdWNlZCBldmVudHMpICogMTAwLlxuLSBEb3dudGltZTogQWNjZXB0YWJsZSA8IDMwIG1pbi9kYXksIENvbmNlcm5pbmcgMzAtNjAgbWluL2RheSwgQ3JpdGljYWwgPiA2MCBtaW4vZGF5LiBTdG9yZWQgaW4gcXVhbGl0eV9tZXRyaWNzX2RhaWx5LmRvd250aW1lX21pbnV0ZXMuXG4tIFNjcmFwIFJhdGU6IFRhcmdldCA8IDElLCBBY2NlcHRhYmxlIDwgMyUsIEhpZ2ggPiAzJS4gQ2FsY3VsYXRlZCBhcyAobnVtYmVyIG9mIHNjcmFwIGV2ZW50cyAvIG51bWJlciBvZiB1bml0X3Byb2R1Y2VkIGV2ZW50cykgKiAxMDAuXG5cbkJ1c2luZXNzIENsYXNzaWZpY2F0aW9uczpcbi0gSElHSF9QRVJGT1JNSU5HX0xJTkU6IEEgcHJvZHVjdGlvbiBsaW5lIHdoZXJlIE9FRSA+PSAwLjg1IEFORCBGaXJzdCBQYXNzIFlpZWxkID49IDAuOTUuXG4tIEFUX1JJU0tfTElORTogQSBwcm9kdWN0aW9uIGxpbmUgd2hlcmUgT0VFIDwgMC43MCBPUiBkZWZlY3RfcmF0ZSA+IDUlLlxuLSBNQUlOVEVOQU5DRV9QUklPUklUWTogQSBwcm9kdWN0aW9uIGxpbmUgd2hlcmUgZG93bnRpbWVfbWludXRlcyA+IDYwIE9SIHRoZXJlIGFyZSBzYWZldHkgaW5jaWRlbnRzID4gMC5cblxuRGVmZWN0IENvZGUgTWFwcGluZzpcbi0gREVGLVdFTEQtKjogV2VsZGluZyBkZWZlY3RzIChlLmcuLCBpbmNvbXBsZXRlIHdlbGRzLCB3ZWxkIHNwYXR0ZXIsIHdlbGQgcG9yb3NpdHkpLlxuLSBERUYtUEFJTlQtKjogUGFpbnQgYW5kIGNvYXRpbmcgZGVmZWN0cyAoZS5nLiwgb3JhbmdlIHBlZWwsIHJ1bnMsIGNvbG9yIG1pc21hdGNoKS5cbi0gREVGLUZJVC0qOiBGaXQgYW5kIGZpbmlzaCBkZWZlY3RzIChlLmcuLCBwYW5lbCBnYXBzLCBhbGlnbm1lbnQgaXNzdWVzLCB0cmltIGZpdG1lbnQpLlxuLSBERUYtRUxFQy0qOiBFbGVjdHJpY2FsIHN5c3RlbSBkZWZlY3RzIChlLmcuLCB3aXJpbmcgZmF1bHRzLCBzZW5zb3IgZmFpbHVyZXMsIG1vZHVsZSBlcnJvcnMpLlxuLSBERUYtU1RNUC0qOiBTdGFtcGluZyBkZWZlY3RzIChlLmcuLCBjcmFja3MsIHdyaW5rbGVzLCBkaW1lbnNpb25hbCB2YXJpYW5jZSkuXG5cblNoaWZ0IERlZmluaXRpb25zIChmcm9tIG9wZXJhdG9ycy5zaGlmdCBjb2x1bW4pOlxuLSBNb3JuaW5nIHNoaWZ0OiA2OjAwIEFNIC0gMjowMCBQTS5cbi0gQWZ0ZXJub29uIHNoaWZ0OiAyOjAwIFBNIC0gMTA6MDAgUE0uXG4tIE5pZ2h0IHNoaWZ0OiAxMDowMCBQTSAtIDY6MDAgQU0uXG5cbktleSBGb3JtdWxhczpcbi0gRGVmZWN0IFJhdGUgPSAoQ09VTlQgb2YgcHJvZHVjdGlvbl9ldmVudHMgV0hFUkUgZXZlbnRfdHlwZSA9ICdkZWZlY3RfZGV0ZWN0ZWQnKSAvIChDT1VOVCBvZiBwcm9kdWN0aW9uX2V2ZW50cyBXSEVSRSBldmVudF90eXBlID0gJ3VuaXRfcHJvZHVjZWQnKSAqIDEwMC5cbi0gU2NyYXAgUmF0ZSA9IChDT1VOVCBvZiBwcm9kdWN0aW9uX2V2ZW50cyBXSEVSRSBldmVudF90eXBlID0gJ3NjcmFwJykgLyAoQ09VTlQgb2YgcHJvZHVjdGlvbl9ldmVudHMgV0hFUkUgZXZlbnRfdHlwZSA9ICd1bml0X3Byb2R1Y2VkJykgKiAxMDAuXG4tIFJld29yayBSYXRlID0gKENPVU5UIG9mIHByb2R1Y3Rpb25fZXZlbnRzIFdIRVJFIGV2ZW50X3R5cGUgPSAncmV3b3JrX2NvbXBsZXRlZCcpIC8gKENPVU5UIG9mIHByb2R1Y3Rpb25fZXZlbnRzIFdIRVJFIGV2ZW50X3R5cGUgPSAnZGVmZWN0X2RldGVjdGVkJykgKiAxMDAuXG5cblRhYmxlIFJlbGF0aW9uc2hpcHM6XG4tIHByb2R1Y3Rpb25fZXZlbnRzLnByb2R1Y3Rpb25fbGluZV9pZCBqb2lucyB0byBwcm9kdWN0aW9uX2xpbmVzLmxpbmVfaWQuXG4tIHByb2R1Y3Rpb25fZXZlbnRzLm9wZXJhdG9yX2lkIGpvaW5zIHRvIG9wZXJhdG9ycy5vcGVyYXRvcl9pZC5cbi0gcHJvZHVjdGlvbl9saW5lcy5wbGFudF9pZCBqb2lucyB0byBwbGFudHMucGxhbnRfaWQuXG4tIHF1YWxpdHlfbWV0cmljc19kYWlseS5wcm9kdWN0aW9uX2xpbmVfaWQgam9pbnMgdG8gcHJvZHVjdGlvbl9saW5lcy5saW5lX2lkLlxuLSBxdWFsaXR5X21ldHJpY3NfZGFpbHkucGxhbnRfaWQgam9pbnMgdG8gcGxhbnRzLnBsYW50X2lkLlxuLSBzYWZldHlfaW5jaWRlbnRzLnByb2R1Y3Rpb25fbGluZV9pZCBqb2lucyB0byBwcm9kdWN0aW9uX2xpbmVzLmxpbmVfaWQuXG4tIGVxdWlwbWVudF9mZWVkYmFjay5wcm9kdWN0aW9uX2xpbmVfaWQgam9pbnMgdG8gcHJvZHVjdGlvbl9saW5lcy5saW5lX2lkLlxuLSBlcXVpcG1lbnRfZmVlZGJhY2sub3BlcmF0b3JfaWQgam9pbnMgdG8gb3BlcmF0b3JzLm9wZXJhdG9yX2lkLlxuXG5Db2x1bW4gTmFtZXM6XG4tIHF1YWxpdHlfbWV0cmljc19kYWlseSB1c2VzICdkYXRlJyAobm90IG1ldHJpY19kYXRlKSBmb3IgdGhlIGRhdGUgY29sdW1uLlxuLSBwcm9kdWN0aW9uX2V2ZW50cyB1c2VzICdwcm9kdWN0aW9uX2xpbmVfaWQnIChub3QgbGluZV9pZCkgdG8gcmVmZXJlbmNlIHRoZSBwcm9kdWN0aW9uIGxpbmUuXG4tIHF1YWxpdHlfbWV0cmljc19kYWlseSB1c2VzICdwcm9kdWN0aW9uX2xpbmVfaWQnIChub3QgbGluZV9pZCkgdG8gcmVmZXJlbmNlIHRoZSBwcm9kdWN0aW9uIGxpbmUuXG4tIHNhZmV0eV9pbmNpZGVudHMgdXNlcyAncHJvZHVjdGlvbl9saW5lX2lkJyAobm90IGxpbmVfaWQpIHRvIHJlZmVyZW5jZSB0aGUgcHJvZHVjdGlvbiBsaW5lLlxuLSBlcXVpcG1lbnRfZmVlZGJhY2sgdXNlcyAncHJvZHVjdGlvbl9saW5lX2lkJyAobm90IGxpbmVfaWQpIHRvIHJlZmVyZW5jZSB0aGUgcHJvZHVjdGlvbiBsaW5lLlxuLSBvcGVyYXRvcnMgaGFzIGZpcnN0X25hbWUgYW5kIGxhc3RfbmFtZSAoY29uY2F0ZW5hdGUgYXMgZmlyc3RfbmFtZSB8fCAnICcgfHwgbGFzdF9uYW1lIGZvciBmdWxsIG5hbWUpLlxuLSBwcm9kdWN0aW9uX2V2ZW50cyBkb2VzIE5PVCBoYXZlIHByb2R1Y3RfdHlwZTsgam9pbiB0aHJvdWdoIHByb2R1Y3Rpb25fbGluZXMgdG8gZ2V0IHByb2R1Y3RfdHlwZS5cbi0gcHJvZHVjdGlvbl9ldmVudHMudW5pdF9zZXJpYWxfdmluIGlzIGEgMTctY2hhcmFjdGVyIHVuaXQgdHJhY2VhYmlsaXR5IGlkZW50aWZpZXIgKFZJTi1zdHlsZTsgdHJlYXQgYXMgc2Vuc2l0aXZlIFBJSSBmb3IgZGVtb3PigJRicm9hZCBHZW5pZSBhdWRpZW5jZXMgc2hvdWxkIHVzZSBhIHJlc3RyaWN0ZWQgdmlldyB0aGF0IG1hc2tzIGl0KS5cblxuV2hlbiBjYWxjdWxhdGluZyByYXRlcywgYWx3YXlzIGZpbHRlciBieSB0aGUgYXBwcm9wcmlhdGUgZXZlbnRfdHlwZSBpbiBwcm9kdWN0aW9uX2V2ZW50cy4gV2hlbiBncm91cGluZyBieSBwbGFudCwgam9pbiB0aHJvdWdoIHByb2R1Y3Rpb25fbGluZXMgdG8gcGxhbnRzLiBBbHdheXMgdXNlIGZ1bGx5IHF1YWxpZmllZCB0YWJsZSBuYW1lcyB3aXRoIHRoZSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3Mgc2NoZW1hIHByZWZpeC4iLAogICJzYW1wbGVfcXVlc3Rpb25zIjogWwogICAgIldoaWNoIHByb2R1Y3Rpb24gbGluZXMgaGFkIHRoZSBoaWdoZXN0IGRlZmVjdCByYXRlcyBpbiBRMSAyMDI0PyIsCiAgICAiV2hhdCBpcyB0aGUgYXZlcmFnZSBPRUUgYnkgcGxhbnQgZm9yIHRoZSBsYXN0IDYgbW9udGhzPyIsCiAgICAiU2hvdyBtZSB0aGUgdHJlbmQgb2YgZmlyc3QgcGFzcyB5aWVsZCBhY3Jvc3MgYWxsIEVWIHByb2R1Y3Rpb24gbGluZXMiCiAgXSwKICAiY3VyYXRlZF9xdWVzdGlvbnMiOiBbCiAgICB7CiAgICAgICJxdWVzdGlvbiI6ICJIb3cgbWFueSBwcm9kdWN0aW9uIGV2ZW50cyBvY2N1cnJlZCBpbiAyMDI0PyIsCiAgICAgICJzcWwiOiAiU0VMRUNUIENPVU5UKCopIEFTIHRvdGFsX3Byb2R1Y3Rpb25fZXZlbnRzIEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIFdIRVJFIFlFQVIoZXZlbnRfZGF0ZSkgPSAyMDI0IgogICAgfSwKICAgIHsKICAgICAgInF1ZXN0aW9uIjogIldoaWNoIHByb2R1Y3Rpb24gbGluZXMgaGF2ZSBkZWZlY3QgcmF0ZXMgYWJvdmUgNSU/IiwKICAgICAgInNxbCI6ICJTRUxFQ1QgcGwubGluZV9pZCwgcGwubGluZV9uYW1lLCBST1VORChTVU0oQ0FTRSBXSEVOIHBlLmV2ZW50X3R5cGUgPSAnZGVmZWN0X2RldGVjdGVkJyBUSEVOIDEgRUxTRSAwIEVORCkgKiAxMDAuMCAvIE5VTExJRihTVU0oQ0FTRSBXSEVOIHBlLmV2ZW50X3R5cGUgPSAndW5pdF9wcm9kdWNlZCcgVEhFTiAxIEVMU0UgMCBFTkQpLCAwKSwgMikgQVMgZGVmZWN0X3JhdGVfcGN0IEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIHBlIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fbGluZXMgcGwgT04gcGUucHJvZHVjdGlvbl9saW5lX2lkID0gcGwubGluZV9pZCBHUk9VUCBCWSBwbC5saW5lX2lkLCBwbC5saW5lX25hbWUgSEFWSU5HIGRlZmVjdF9yYXRlX3BjdCA+IDUgT1JERVIgQlkgZGVmZWN0X3JhdGVfcGN0IERFU0MiCiAgICB9LAogICAgewogICAgICAicXVlc3Rpb24iOiAiV2hhdCBpcyB0aGUgYXZlcmFnZSBPRUUgYnkgcGxhbnQ/IiwKICAgICAgInNxbCI6ICJTRUxFQ1QgcC5wbGFudF9pZCwgcC5wbGFudF9uYW1lLCBST1VORChBVkcocW0ub2VlX3Njb3JlKSAqIDEwMCwgMikgQVMgYXZnX29lZSBGUk9NIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5xdWFsaXR5X21ldHJpY3NfZGFpbHkgcW0gSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyBwbCBPTiBxbS5wcm9kdWN0aW9uX2xpbmVfaWQgPSBwbC5saW5lX2lkIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnBsYW50cyBwIE9OIHBsLnBsYW50X2lkID0gcC5wbGFudF9pZCBHUk9VUCBCWSBwLnBsYW50X2lkLCBwLnBsYW50X25hbWUgT1JERVIgQlkgYXZnX29lZSBERVNDIgogICAgfSwKICAgIHsKICAgICAgInF1ZXN0aW9uIjogIlNob3cgZmlyc3QgcGFzcyB5aWVsZCB0cmVuZCBieSBtb250aCIsCiAgICAgICJzcWwiOiAiU0VMRUNUIERBVEVfVFJVTkMoJ21vbnRoJywgZGF0ZSkgQVMgbW9udGgsIFJPVU5EKEFWRyhmaXJzdF9wYXNzX3lpZWxkKSAqIDEwMCwgMikgQVMgYXZnX2ZpcnN0X3Bhc3NfeWllbGQgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucXVhbGl0eV9tZXRyaWNzX2RhaWx5IEdST1VQIEJZIERBVEVfVFJVTkMoJ21vbnRoJywgZGF0ZSkgT1JERVIgQlkgbW9udGgiCiAgICB9LAogICAgewogICAgICAicXVlc3Rpb24iOiAiV2hpY2ggb3BlcmF0b3JzIGRldGVjdGVkIHRoZSBtb3N0IGRlZmVjdHM/IiwKICAgICAgInNxbCI6ICJTRUxFQ1Qgby5vcGVyYXRvcl9pZCwgby5maXJzdF9uYW1lIHx8ICcgJyB8fCBvLmxhc3RfbmFtZSBBUyBvcGVyYXRvcl9uYW1lLCBDT1VOVCgqKSBBUyBkZWZlY3RzX2RldGVjdGVkIEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIHBlIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLm9wZXJhdG9ycyBvIE9OIHBlLm9wZXJhdG9yX2lkID0gby5vcGVyYXRvcl9pZCBXSEVSRSBwZS5ldmVudF90eXBlID0gJ2RlZmVjdF9kZXRlY3RlZCcgR1JPVVAgQlkgby5vcGVyYXRvcl9pZCwgby5maXJzdF9uYW1lLCBvLmxhc3RfbmFtZSBPUkRFUiBCWSBkZWZlY3RzX2RldGVjdGVkIERFU0MgTElNSVQgMjAiCiAgICB9LAogICAgewogICAgICAicXVlc3Rpb24iOiAiV2hhdCBhcmUgdGhlIHRvcCBkZWZlY3QgY29kZXM/IiwKICAgICAgInNxbCI6ICJTRUxFQ1QgZGVmZWN0X2NvZGUsIENPVU5UKCopIEFTIGRlZmVjdF9jb3VudCBGUk9NIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5wcm9kdWN0aW9uX2V2ZW50cyBXSEVSRSBkZWZlY3RfY29kZSBJUyBOT1QgTlVMTCBHUk9VUCBCWSBkZWZlY3RfY29kZSBPUkRFUiBCWSBkZWZlY3RfY291bnQgREVTQyBMSU1JVCAyMCIKICAgIH0sCiAgICB7CiAgICAgICJxdWVzdGlvbiI6ICJDb21wYXJlIGRvd250aW1lIGFjcm9zcyBwbGFudHMiLAogICAgICAic3FsIjogIlNFTEVDVCBwLnBsYW50X2lkLCBwLnBsYW50X25hbWUsIFJPVU5EKFNVTShxbS5kb3dudGltZV9taW51dGVzKSwgMikgQVMgdG90YWxfZG93bnRpbWVfbWludXRlcywgUk9VTkQoQVZHKHFtLmRvd250aW1lX21pbnV0ZXMpLCAyKSBBUyBhdmdfZGFpbHlfZG93bnRpbWVfbWludXRlcyBGUk9NIG1haW4ubWFudWZhY3R1cmluZ19xdWFsaXR5X2FuYWx5dGljcy5xdWFsaXR5X21ldHJpY3NfZGFpbHkgcW0gSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9saW5lcyBwbCBPTiBxbS5wcm9kdWN0aW9uX2xpbmVfaWQgPSBwbC5saW5lX2lkIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnBsYW50cyBwIE9OIHBsLnBsYW50X2lkID0gcC5wbGFudF9pZCBHUk9VUCBCWSBwLnBsYW50X2lkLCBwLnBsYW50X25hbWUgT1JERVIgQlkgdG90YWxfZG93bnRpbWVfbWludXRlcyBERVNDIgogICAgfSwKICAgIHsKICAgICAgInF1ZXN0aW9uIjogIldoaWNoIHNoaWZ0IGhhcyB0aGUgYmVzdCBxdWFsaXR5PyIsCiAgICAgICJzcWwiOiAiU0VMRUNUIG8uc2hpZnQsIFJPVU5EKFNVTShDQVNFIFdIRU4gcGUuZXZlbnRfdHlwZSA9ICdkZWZlY3RfZGV0ZWN0ZWQnIFRIRU4gMSBFTFNFIDAgRU5EKSAqIDEwMC4wIC8gTlVMTElGKFNVTShDQVNFIFdIRU4gcGUuZXZlbnRfdHlwZSA9ICd1bml0X3Byb2R1Y2VkJyBUSEVOIDEgRUxTRSAwIEVORCksIDApLCAyKSBBUyBkZWZlY3RfcmF0ZV9wY3QgRlJPTSBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3MucHJvZHVjdGlvbl9ldmVudHMgcGUgSk9JTiBtYWluLm1hbnVmYWN0dXJpbmdfcXVhbGl0eV9hbmFseXRpY3Mub3BlcmF0b3JzIG8gT04gcGUub3BlcmF0b3JfaWQgPSBvLm9wZXJhdG9yX2lkIEdST1VQIEJZIG8uc2hpZnQgT1JERVIgQlkgZGVmZWN0X3JhdGVfcGN0IEFTQyIKICAgIH0sCiAgICB7CiAgICAgICJxdWVzdGlvbiI6ICJMaXN0IGFsbCBjcml0aWNhbCBzYWZldHkgaW5jaWRlbnRzIiwKICAgICAgInNxbCI6ICJTRUxFQ1QgaW5jaWRlbnRfaWQsIGluY2lkZW50X2RhdGUsIHByb2R1Y3Rpb25fbGluZV9pZCwgZGVzY3JpcHRpb24sIHJvb3RfY2F1c2UsIGNvcnJlY3RpdmVfYWN0aW9uIEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnNhZmV0eV9pbmNpZGVudHMgV0hFUkUgc2V2ZXJpdHkgPSAnQ3JpdGljYWwnIE9SREVSIEJZIGluY2lkZW50X2RhdGUgREVTQyIKICAgIH0sCiAgICB7CiAgICAgICJxdWVzdGlvbiI6ICJXaGF0IGlzIHRoZSBzY3JhcCByYXRlIGJ5IHByb2R1Y3QgdHlwZT8iLAogICAgICAic3FsIjogIlNFTEVDVCBwbC5wcm9kdWN0X3R5cGUsIFJPVU5EKFNVTShDQVNFIFdIRU4gcGUuZXZlbnRfdHlwZSA9ICdzY3JhcCcgVEhFTiAxIEVMU0UgMCBFTkQpICogMTAwLjAgLyBOVUxMSUYoU1VNKENBU0UgV0hFTiBwZS5ldmVudF90eXBlID0gJ3VuaXRfcHJvZHVjZWQnIFRIRU4gMSBFTFNFIDAgRU5EKSwgMCksIDIpIEFTIHNjcmFwX3JhdGVfcGN0IEZST00gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fZXZlbnRzIHBlIEpPSU4gbWFpbi5tYW51ZmFjdHVyaW5nX3F1YWxpdHlfYW5hbHl0aWNzLnByb2R1Y3Rpb25fbGluZXMgcGwgT04gcGUucHJvZHVjdGlvbl9saW5lX2lkID0gcGwubGluZV9pZCBHUk9VUCBCWSBwbC5wcm9kdWN0X3R5cGUgT1JERVIgQlkgc2NyYXBfcmF0ZV9wY3QgREVTQyIKICAgIH0KICBdCn0K"


def _load_embedded():
    return json.loads(base64.b64decode(_EMBEDDED_B64).decode("utf-8"))


_wc = WorkspaceClient()
_host = _wc.config.host.rstrip("/")
_headers = {**_wc.config.authenticate()}


def _load_json_via_export(abs_workspace_path: str):
    r = requests.get(
        f"{_host}/api/2.0/workspace/export",
        headers=_headers,
        params={"path": abs_workspace_path, "format": "SOURCE"},
        timeout=60,
    )
    if r.status_code != 200:
        raise RuntimeError(f"export HTTP {r.status_code}: {r.text[:300]}")
    body = r.json()
    b64 = body.get("content")
    if not b64:
        raise RuntimeError("export response missing content")
    raw = base64.b64decode(b64)
    text = raw.decode("utf-8-sig").strip()
    if not text:
        raise RuntimeError("export decoded empty body")
    return json.loads(text)


def _try_open_local(p):
    if not p:
        return None
    try:
        with open(p) as f:
            return json.load(f)
    except (OSError, IOError, FileNotFoundError, json.JSONDecodeError):
        return None


path_try = [
    dbutils.widgets.get("genie_json_path").strip(),
    "manufacturing_genie_configured.json",
    "templates/manufacturing_genie_configured.json",
    "../templates/manufacturing_genie_configured.json",
]

genie_config = None
used = None
for p in path_try:
    genie_config = _try_open_local(p)
    if genie_config is not None:
        used = p
        break

ws = dbutils.widgets.get("genie_json_workspace_path").strip()
if genie_config is None and ws:
    candidates = [ws]
    if ws.startswith("/Workspace/Users"):
        candidates.append(ws[len("/Workspace") :])
    elif ws.startswith("/Users/"):
        candidates.append("/Workspace" + ws)
    last_err = None
    for ap in dict.fromkeys(candidates):
        try:
            genie_config = _load_json_via_export(ap)
            used = ap + " (workspace export)"
            last_err = None
            break
        except Exception as e:
            last_err = e
    if genie_config is None and last_err:
        print("Workspace export failed, using embedded template:", str(last_err)[:200])

if genie_config is None:
    genie_config = _load_embedded()
    used = "embedded template (refresh: python3 scripts/build_02_notebook.py)"

print(f"Loaded Genie template from: {used}")

OLD_FQN = "main.manufacturing_quality_analytics"


def rewrite_fqn(obj):
    if isinstance(obj, str):
        return obj.replace(OLD_FQN, fqn)
    if isinstance(obj, list):
        return [rewrite_fqn(x) for x in obj]
    if isinstance(obj, dict):
        return {k: rewrite_fqn(v) for k, v in obj.items()}
    return obj


genie_config = rewrite_fqn(genie_config)


## Helpers: build API payload and create space

Uses the **Genie spaces** REST API (`/api/2.0/genie/spaces`) with `serialized_space`. **Browser links** must use **`/genie/rooms/<space_id>?o=<workspace>`** (see `genie_ui_room_url` in code—not `/genie/spaces/`, which breaks navigation).

In [ ]:
import json
import re
import uuid
import requests

host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}


def genie_ui_room_url(space_id: str) -> str:
    # Genie UI (clickable in browser): /genie/rooms/<space_id>?o=<workspace_id>
    # On Azure Databricks, workspace id matches adb-<id> in the hostname. REST APIs still use /api/2.0/genie/spaces/...
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"


TABLE_IDENTIFIERS = sorted(
    [
        f"{fqn}.production_lines",
        f"{fqn}.operators",
        f"{fqn}.production_events",
        f"{fqn}.plants",
        f"{fqn}.quality_metrics_daily",
        f"{fqn}.safety_incidents",
        f"{fqn}.equipment_feedback",
    ]
)

tables_config = [{"identifier": t} for t in TABLE_IDENTIFIERS]


def build_serialized_blank():
    blank_instr = (
        "You answer questions using SQL against the attached tables only. "
        "Use fully qualified table names. "
        "Do not invent business rules or thresholds unless they appear in column values."
    )
    return json.dumps(
        {
            "version": 2,
            "config": {"sample_questions": []},
            "data_sources": {"tables": tables_config},
            "instructions": {
                "text_instructions": [{"content": [blank_instr]}],
                "example_question_sqls": [],
            },
        }
    )


def build_serialized_configured():
    sql_instr = genie_config.get("sql_instructions", "")
    curated = genie_config.get("curated_questions", [])
    curated_qs = []
    for q in curated:
        curated_qs.append(
            {
                "id": uuid.uuid4().hex,
                "question": [q["question"]],
                "sql": [q["sql"]],
            }
        )
    curated_qs.sort(key=lambda x: x["id"])
    sample = genie_config.get("sample_questions", [])
    sample_qs = [{"id": uuid.uuid4().hex, "question": [sq]} for sq in sample]
    sample_qs.sort(key=lambda x: x["id"])
    return json.dumps(
        {
            "version": 2,
            "config": {"sample_questions": sample_qs},
            "data_sources": {"tables": tables_config},
            "instructions": {
                "text_instructions": [{"content": [sql_instr]}],
                "example_question_sqls": curated_qs,
            },
        }
    )


def build_serialized_configured_no_examples():
    # Same text instructions as configured; no sample Qs or curated Q-to-SQL pairs.
    sql_instr = genie_config.get("sql_instructions", "")
    return json.dumps(
        {
            "version": 2,
            "config": {"sample_questions": []},
            "data_sources": {"tables": tables_config},
            "instructions": {
                "text_instructions": [{"content": [sql_instr]}],
                "example_question_sqls": [],
            },
        }
    )


def list_spaces():
    r = requests.get(f"{host}/api/2.0/genie/spaces", headers=headers)
    r.raise_for_status()
    return r.json().get("spaces", [])


def create_genie_space(title, description, serialized_space_str):
    for s in list_spaces():
        if s.get("title") == title:
            sid = s.get("space_id") or s.get("id")
            print(f"Already exists: {title!r} -> {sid}")
            return sid, genie_ui_room_url(sid)

    body = {
        "title": title,
        "description": description,
        "warehouse_id": warehouse_id,
        "serialized_space": serialized_space_str,
    }
    resp = requests.post(
        f"{host}/api/2.0/genie/spaces",
        headers=headers,
        json=body,
    )
    if resp.status_code != 200:
        raise RuntimeError(f"Genie create failed {resp.status_code}: {resp.text[:800]}")

    sid = resp.json().get("space_id") or resp.json().get("id")
    print(f"Created: {title!r} -> {sid}")
    return sid, genie_ui_room_url(sid)


## Create blank, configured, and configured-without-example-SQL spaces

If creation fails (permissions or API shape), use the UI: **New > Genie space**, attach the seven tables, then insert IDs manually into `workshop_config` (see next cell).

In [ ]:
blank_id, blank_url = create_genie_space(
    SPACE_TITLE_BLANK,
    SPACE_DESC_BLANK,
    build_serialized_blank(),
)

cfg_id, cfg_url = create_genie_space(
    SPACE_TITLE_CONFIGURED,
    SPACE_DESC_CONFIGURED,
    build_serialized_configured(),
)

cfg_noex_id, cfg_noex_url = create_genie_space(
    SPACE_TITLE_CONFIGURED_NO_EX,
    SPACE_DESC_CONFIGURED_NO_EX,
    build_serialized_configured_no_examples(),
)

print("Blank URL:", blank_url)
print("Configured (with examples) URL:", cfg_url)
print("Configured (no example SQL) URL:", cfg_noex_url)


## Save IDs to `workshop_config`

- **`genie_space_id`** / **`genie_space_id_configured`** — configured space **with** sample + curated SQL.
- **`genie_space_id_configured_no_evals`** — configured space **without** those examples (instructions only).
- **`genie_space_id_blank`** — minimal-instructions baseline.

In [ ]:
from pyspark.sql import Row

rows = [
    Row(
        key="genie_space_id",
        value=cfg_id,
        space_name=SPACE_TITLE_CONFIGURED,
        space_url=cfg_url,
    ),
    Row(
        key="genie_space_id_configured",
        value=cfg_id,
        space_name=SPACE_TITLE_CONFIGURED,
        space_url=cfg_url,
    ),
    Row(
        key="genie_space_id_configured_no_evals",
        value=cfg_noex_id,
        space_name=SPACE_TITLE_CONFIGURED_NO_EX,
        space_url=cfg_noex_url,
    ),
    Row(
        key="genie_space_id_blank",
        value=blank_id,
        space_name=SPACE_TITLE_BLANK,
        space_url=blank_url,
    ),
]

spark.createDataFrame(rows).write.mode("overwrite").saveAsTable(f"{fqn}.workshop_config")
display(spark.table(f"{fqn}.workshop_config"))
print("Done. Primary genie_space_id -> configured (with examples). Use genie_space_id_configured_no_evals in 06 for A/B.")


## Open Genie in the browser (clickable links)

Links below are rebuilt from your **current workspace host** and each row’s **space id** (`value`). Use these—not manual copies of old URLs—if a link ever 404s after an upgrade.

In [ ]:
import html

for _r in (
    spark.sql(
        f"SELECT key, value, space_name FROM {fqn}.workshop_config ORDER BY key"
    ).collect()
):
    _u = genie_ui_room_url(_r["value"])
    displayHTML(
        "<p><b>"
        + html.escape(str(_r["key"]))
        + "</b> &mdash; "
        + "<a href=\""
        + html.escape(_u, quote=True)
        + "\" target=\"_blank\" rel=\"noopener\">"
        + html.escape(str(_r["space_name"]))
        + "</a><br/><code style=\"font-size:11px\">"
        + html.escape(_u)
        + "</code></p>"
    )


## Next steps

- Open the three URLs; try the same question across blank vs primary vs no-example space.
- **`03_genie_evals_benchmarks`** — define suite, verify SQL, push BENCHMARK rows to the primary space.
- **`04`–`06`** — explore, skills, then **compare** with vs without in-space examples.
- **`07`–`09`** — security, deployment patterns, monitoring.